In [3]:
import os
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

from pydantic_settings import BaseSettings, SettingsConfigDict
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
from openai import OpenAI

# --- 1. 환경 설정 ---
class Config(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore"
    )
    postgres_user: str
    postgres_password: str
    postgres_db: str
    postgres_host: str = "localhost"
    postgres_port: int = 9010
    openai_api_key: str

settings = Config()

# --- 2. 데이터 구조 (LlamaIndex 스타일) ---
@dataclass
class Document:
    text: str
    metadata: Optional[Dict[str, Any]] = None
    id: Optional[str] = None  # 자동 생성되거나 지정 가능


In [4]:

# --- 3. PGVectorStore 클래스 정의 ---
class PGVectorStore:
    def __init__(self, table_name: str = "hybrid_documents", vector_dim: int = 1536):
        """
        초기화 및 DB 연결, 테이블/인덱스 생성
        vector_dim: OpenAI text-embedding-3-small 기준 1536
        """
        self.table_name = table_name
        self.vector_dim = vector_dim
        
        # DB URL 구성
        self.db_url = f"postgresql://{settings.postgres_user}:{settings.postgres_password}@{settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}"
        self.engine = create_engine(self.db_url)
        self.SessionLocal = sessionmaker(bind=self.engine)
        
        # OpenAI Client 초기화
        self.aclient = OpenAI(api_key=settings.openai_api_key)

        # DB 초기화 (테이블 및 인덱스 생성)
        self._initialize_db()

    def _initialize_db(self):
        """테이블과 Vector/BM25 인덱스를 생성합니다."""
        session = self.SessionLocal()
        try:
            # 1. 확장 기능 활성화 확인
            session.execute(text("CREATE EXTENSION IF NOT EXISTS vector;"))
            session.execute(text("CREATE EXTENSION IF NOT EXISTS pg_textsearch;"))
            
            # 2. 테이블 생성
            # id: Serial, content: 텍스트, embedding: 벡터, metadata: JSONB
            create_table_sql = text(f"""
                CREATE TABLE IF NOT EXISTS {self.table_name} (
                    id SERIAL PRIMARY KEY,
                    content TEXT NOT NULL,
                    embedding vector({self.vector_dim}),
                    metadata JSONB DEFAULT '{{}}'::jsonb
                );
            """)
            session.execute(create_table_sql)

            # 3. HNSW 인덱스 생성 (Vector Search용)
            # vector_ops: cosine distance (<=>) 최적화를 위해 vector_cosine_ops 사용 가능 (기본은 L2)
            # 여기서는 HNSW 사용
            create_hnsw_idx = text(f"""
                CREATE INDEX IF NOT EXISTS idx_{self.table_name}_embedding 
                ON {self.table_name} USING hnsw (embedding vector_cosine_ops);
            """)
            session.execute(create_hnsw_idx)

            # 4. BM25 인덱스 생성 (Keyword Search용)
            # text_config='public.korean' -> 앞서 정의한 Mecab 구성 사용
            create_bm25_idx = text(f"""
                CREATE INDEX IF NOT EXISTS idx_{self.table_name}_bm25 
                ON {self.table_name} USING bm25 (content) 
                WITH (text_config='public.korean');
            """)
            session.execute(create_bm25_idx)
            
            session.commit()
            print(f"✅ DB 초기화 완료: 테이블 '{self.table_name}' 및 인덱스 준비됨.")
            
        except Exception as e:
            session.rollback()
            print(f"❌ DB 초기화 중 오류: {e}")
            raise e
        finally:
            session.close()

    def _get_embedding(self, text_input: str) -> List[float]:
        """OpenAI를 사용하여 임베딩 생성"""
        text_input = text_input.replace("\n", " ")
        response = self.aclient.embeddings.create(
            input=[text_input],
            model="text-embedding-3-small" # 또는 text-embedding-ada-002
        )
        return response.data[0].embedding

    def add_documents(self, documents: List[Document]):
        """문서 리스트를 받아 임베딩 후 DB에 저장"""
        session = self.SessionLocal()
        try:
            for doc in documents:
                # 1. 임베딩 생성
                embedding_vec = self._get_embedding(doc.text)
                
                # 2. SQL 파라미터 바인딩으로 삽입
                insert_sql = text(f"""
                    INSERT INTO {self.table_name} (content, embedding, metadata)
                    VALUES (:content, :embedding, :metadata)
                """)
                
                session.execute(insert_sql, {
                    "content": doc.text,
                    "embedding": str(embedding_vec), # pgvector는 배열을 문자열이나 리스트로 받음
                    "metadata": json.dumps(doc.metadata) if doc.metadata else "{}"
                })
            
            session.commit()
            print(f"✅ {len(documents)}개의 문서를 저장했습니다.")
            
        except Exception as e:
            session.rollback()
            print(f"❌ 문서 저장 중 오류: {e}")
        finally:
            session.close()

    def hybrid_search(self, query: str, top_k: int = 5, rrf_k: int = 60):
        """
        RRF(Reciprocal Rank Fusion)를 사용한 하이브리드 검색
        
        Args:
            query (str): 검색어
            top_k (int): 최종 반환할 문서 개수
            rrf_k (int): RRF 상수 (보통 60 사용)
        """
        session = self.SessionLocal()
        results = []
        try:
            # 1. 쿼리 임베딩 생성
            query_embedding = self._get_embedding(query)

            # 2. Hybrid Search SQL 작성
            # CTE 설명:
            # - semantic_search: HNSW 인덱스를 사용하여 벡터 거리가 가까운 순서대로 순위(rank) 매김
            # - keyword_search: BM25 인덱스를 사용하여 텍스트 관련도가 높은 순서대로 순위(rank) 매김
            # - 메인 쿼리: 두 결과의 순위를 RRF 공식으로 합산하여 정렬
            
            sql = text(f"""
                WITH semantic_search AS (
                    SELECT 
                        id, 
                        content, 
                        metadata,
                        ROW_NUMBER() OVER (ORDER BY embedding <=> :query_vec) as rank
                    FROM {self.table_name}
                    ORDER BY embedding <=> :query_vec
                    LIMIT :limit_count
                ),
                keyword_search AS (
                    SELECT 
                        id, 
                        content, 
                        metadata,
                        ROW_NUMBER() OVER (ORDER BY content <@> :query_text) as rank
                    FROM {self.table_name}
                    ORDER BY content <@> :query_text
                    LIMIT :limit_count
                )
                SELECT 
                    COALESCE(s.id, k.id) as id,
                    COALESCE(s.content, k.content) as content,
                    COALESCE(s.metadata, k.metadata) as metadata,
                    COALESCE(1.0 / (:rrf_k + s.rank), 0.0) + 
                    COALESCE(1.0 / (:rrf_k + k.rank), 0.0) as rrf_score
                FROM semantic_search s
                FULL OUTER JOIN keyword_search k ON s.id = k.id
                ORDER BY rrf_score DESC
                LIMIT :top_k;
            """)

            # 실행
            rows = session.execute(sql, {
                "query_vec": str(query_embedding),
                "query_text": query,
                "limit_count": top_k * 2, # RRF 풀을 위해 요청된 것보다 좀 더 많이 후보군을 뽑습니다
                "rrf_k": rrf_k,
                "top_k": top_k
            }).fetchall()

            # 결과 포맷팅
            print(f"\n🔍 Hybrid Search Result for '{query}':")
            for i, row in enumerate(rows, 1):
                results.append(row)
                print(f"{i}. [Score: {row.rrf_score:.4f}] {row.content[:50]}...")

        except Exception as e:
            print(f"❌ 검색 중 오류: {e}")
        finally:
            session.close()
        
        return results


In [6]:

# --- 4. 실행 예시 ---
# DB Store 초기화
store = PGVectorStore(table_name="my_docs")

    # 예제 데이터 (메타데이터 포함 가능)
sample_docs = [
    Document(text="애플은 아이폰을 만드는 IT 기업입니다.", metadata={"category": "tech"}),
    Document(text="사과는 맛있는 과일이며 비타민이 풍부합니다.", metadata={"category": "food"}),
    Document(text="배는 기관지에 좋은 과일입니다.", metadata={"category": "food"}),
    Document(text="거대한 배가 항구에 정박해 있습니다.", metadata={"category": "transport"}),
    Document(text="머신러닝과 딥러닝은 인공지능의 하위 분야입니다.", metadata={"category": "AI"}),
    Document(text="HNSW 인덱스는 벡터 검색 속도를 높여줍니다.", metadata={"category": "DB"}),
    Document(text="PostgreSQL은 강력한 오픈소스 관계형 데이터베이스입니다.", metadata={"category": "DB"}),
]

# 데이터 추가 (최초 1회만 실행하거나, 중복 체크 로직 추가 필요)
store.add_documents(sample_docs)


✅ DB 초기화 완료: 테이블 'my_docs' 및 인덱스 준비됨.
✅ 7개의 문서를 저장했습니다.


In [7]:
# 검색 테스트
# 1. '배' (과일 vs 선박) -> 키워드는 같지만 의미에 따라 다를 수 있음
result1=store.hybrid_search("항구에 있는 배") 


🔍 Hybrid Search Result for '항구에 있는 배':
1. [Score: 0.0328] 거대한 배가 항구에 정박해 있습니다....
2. [Score: 0.0323] 배는 기관지에 좋은 과일입니다....
3. [Score: 0.0159] HNSW 인덱스는 벡터 검색 속도를 높여줍니다....
4. [Score: 0.0156] PostgreSQL은 강력한 오픈소스 관계형 데이터베이스입니다....
5. [Score: 0.0154] 애플은 아이폰을 만드는 IT 기업입니다....


In [8]:
result1

[(4, '거대한 배가 항구에 정박해 있습니다.', {'category': 'transport'}, Decimal('0.03278688524590163934')),
 (3, '배는 기관지에 좋은 과일입니다.', {'category': 'food'}, Decimal('0.03225806451612903226')),
 (6, 'HNSW 인덱스는 벡터 검색 속도를 높여줍니다.', {'category': 'DB'}, Decimal('0.01587301587301587302')),
 (7, 'PostgreSQL은 강력한 오픈소스 관계형 데이터베이스입니다.', {'category': 'DB'}, Decimal('0.01562500000000000000')),
 (1, '애플은 아이폰을 만드는 IT 기업입니다.', {'category': 'tech'}, Decimal('0.01538461538461538462'))]

In [9]:
# 2. '데이터베이스' -> 벡터(HNSW)와 키워드(Postgres)가 결합되어 검색됨
result2=store.hybrid_search("빠른 데이터베이스 검색")


🔍 Hybrid Search Result for '빠른 데이터베이스 검색':
1. [Score: 0.0328] HNSW 인덱스는 벡터 검색 속도를 높여줍니다....
2. [Score: 0.0323] PostgreSQL은 강력한 오픈소스 관계형 데이터베이스입니다....
3. [Score: 0.0159] 머신러닝과 딥러닝은 인공지능의 하위 분야입니다....
4. [Score: 0.0156] 배는 기관지에 좋은 과일입니다....
5. [Score: 0.0154] 애플은 아이폰을 만드는 IT 기업입니다....


In [10]:
result2

[(6, 'HNSW 인덱스는 벡터 검색 속도를 높여줍니다.', {'category': 'DB'}, Decimal('0.03278688524590163934')),
 (7, 'PostgreSQL은 강력한 오픈소스 관계형 데이터베이스입니다.', {'category': 'DB'}, Decimal('0.03225806451612903226')),
 (5, '머신러닝과 딥러닝은 인공지능의 하위 분야입니다.', {'category': 'AI'}, Decimal('0.01587301587301587302')),
 (3, '배는 기관지에 좋은 과일입니다.', {'category': 'food'}, Decimal('0.01562500000000000000')),
 (1, '애플은 아이폰을 만드는 IT 기업입니다.', {'category': 'tech'}, Decimal('0.01538461538461538462'))]